<a href="https://colab.research.google.com/github/Yeshwanth369/ML-DL-Projects/blob/main/llama_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install  bitsandbytes
!pip install accelerate
!pip install -i https://pypi.org/simple/ bitsandbytes

# After Install The Libraries . Restart the Kernel

Looking in indexes: https://pypi.org/simple/


In [1]:
!pip install pypdf
import pypdf

reader = pypdf.PdfReader("/content/Project_Docs_for_Yeshwanth_Pulapa.pdf")
print(f"Loaded PDF with {len(reader.pages)} pages")

full_text = ""
for page_num, page in enumerate(reader.pages):
  page_text = page.extract_text()
  full_text += f"\n--- Page {page_num + 1} ---\n {page_text}"

print(f"Total characters extracted: {len(full_text):,}")

Loaded PDF with 10 pages
Total characters extracted: 39,611


In [2]:
print(full_text)


--- Page 1 ---
 Intro Script 
 
I’m Yeshwanth Pulapa , an AI/ML Engineer  with hands -on experience designing and deploying 
intelligent systems using Databricks, AWS, and Spark. Most recently, I’ve been working at Databricks 
USA, where I’ve built and productionized machine learning pipelines  for real-time fraud detection 
and customer automation. 
 
One of my key projects involved integrating Kafka streaming, Databricks Structured Streaming, and 
GPT-based chatbots  to detect fraudulent transactions and assist customers automatically. This 
solution processed over a million transactions per day , reducing fraud detection time by 80% and 
cutting manual review efforts by 60% — a really rewarding experience that showed the power of 
combining AI and automation. 
 
Before Databricks, I worked as an AI/ML Engineer at Trigma in India, where I developed predictive 
models, recommendation systems, and lead -scoring algorithms  for clients across Real Estate, 
Healthcare, and E -commerce s

In [3]:
!pip install langchain_text_splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800, # Each chunk will target 800 characters
    chunk_overlap = 150, # Chunks share 150 characters at there boundaries so that the context is not lost
    separators = ["\n\n", "\n", "(?<=\\. )", " ", "",":","  ","   "]

)

chunks = text_splitter.split_text(full_text)

print(f"Created {len(chunks)} chunks")
print(f"Average chunk size : {sum(len(c) for c in chunks)//len(chunks)} characters")


Created 62 chunks
Average chunk size : 744 characters


In [4]:
print(chunks[1])

cutting manual review efforts by 60% — a really rewarding experience that showed the power of 
combining AI and automation. 
 
Before Databricks, I worked as an AI/ML Engineer at Trigma in India, where I developed predictive 
models, recommendation systems, and lead -scoring algorithms  for clients across Real Estate, 
Healthcare, and E -commerce sectors. I used tools like Scikit-learn, XGBoost, LightFM, and AWS 
EC2/S3 to build and deploy scalable ML solutions, improving lead conversions and pricing accuracy. 
 
I recently completed my Master of Science in Artificial Intelligence from the University of North 
Texas, which strengthened my foundation in deep learning, MLOps, and scalable model 
deployment.


In [5]:
print("Chunk previews : ")
print("=="*40)

for i , chunk in enumerate(chunks):
  preview = chunk[:100].replace("\n",' ')
  print(f"chunk {i+1 : 2d} : {len(chunk):4d} chars | {preview}....")

Chunk previews : 
chunk  1 :  772 chars | --- Page 1 ---  Intro Script    I’m Yeshwanth Pulapa , an AI/ML Engineer  with hands -on experience ....
chunk  2 :  714 chars | cutting manual review efforts by 60% — a really rewarding experience that showed the power of  combi....
chunk  3 :  768 chars | Texas, which strengthened my foundation in deep learning, MLOps, and scalable model  deployment.    ....
chunk  4 :  771 chars | Client Overview: Wellspring Financial Services is a mid -sized digital -first banking institution ca....
chunk  5 :  755 chars | engine that could process and analyze more than 1 million transactions daily with sub-second latency....
chunk  6 :  721 chars | identify complex fraud scenarios such as social engineering, identity theft, and synthetic accounts.....
chunk  7 :  710 chars | Additionally, I automated model retraining pipelines  with Databricks Jobs , allowing models to adap....
chunk  8 :  751 chars | milliseconds — but our fraud detection models needed de

In [6]:
# Finding word related chunk, retriving only one chunk for testing
for i , chunk in enumerate(chunks):
  if "Good Morning" in chunk:
    print(f"Chunk {i+1} :")
    print(chunk)
    break
  elif i == len(chunks)-1:
    print("No chunk found")


No chunk found


In [7]:
#Vector Database
# Embedding are key to semantic search
! pip install sentence-transformers chromadb

In [8]:
from sentence_transformers import SentenceTransformer

print("Loading embedding model....")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model is loaded")

test_embedding = embedder.encode("Hello world")
print(test_embedding)
print(f"Embedding size : {len(test_embedding)}")

Loading embedding model....


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Embedding model is loaded
[-3.44772898e-02  3.10231987e-02  6.73497515e-03  2.61090174e-02
 -3.93620022e-02 -1.60302505e-01  6.69240057e-02 -6.44143764e-03
 -4.74505089e-02  1.47588663e-02  7.08753541e-02  5.55275530e-02
  1.91933122e-02 -2.62513384e-02 -1.01095233e-02 -2.69405153e-02
  2.23074164e-02 -2.22266503e-02 -1.49692640e-01 -1.74931008e-02
  7.67624378e-03  5.43523133e-02  3.25445877e-03  3.17259617e-02
 -8.46214145e-02 -2.94059981e-02  5.15956841e-02  4.81240749e-02
 -3.31480149e-03 -5.82792051e-02  4.19693030e-02  2.22107004e-02
  1.28188834e-01 -2.23389678e-02 -1.16562489e-02  6.29283786e-02
 -3.28762978e-02 -9.12260637e-02 -3.11753228e-02  5.26995435e-02
  4.70348373e-02 -8.42030272e-02 -3.00562102e-02 -2.07448434e-02
  9.51777771e-03 -3.72177688e-03  7.34331505e-03  3.93243618e-02
  9.32740495e-02 -3.78857437e-03 -5.27421162e-02 -5.80582432e-02
 -6.86437311e-03  5.28324163e-03  8.28930289e-02  1.93627626e-02
  6.28448091e-03 -1.03307683e-02  9.03238542e-03 -3.76837812e-02

In [9]:
import numpy as np

def cosine_similarity(v1,v2):
  return np.dot(v1,v2)/(np.linalg.norm(v1)*np.linalg.norm(v2))

Test_Phrase_1 = "the world is spherical shape"
Test_Phrase_2 = "the world is flat shape"
Test_Phrase_3 = "the earth is flat shape"
Test_Phrase_4 = "the earth is our world"

emb1 = embedder.encode(Test_Phrase_1)
emb2 = embedder.encode(Test_Phrase_2)
emb3 = embedder.encode(Test_Phrase_3)
emb4 = embedder.encode(Test_Phrase_4)

print(f"Similarity Score for Phrase1 vs Phrase2 : {cosine_similarity(emb1,emb2)}")
print(f"Similarity Score for Phrase2 vs Phrase3 : {cosine_similarity(emb2,emb3)}")
print(f"Similarity Score for Phrase1 vs Phrase3 : {cosine_similarity(emb1,emb3)}")
print(f"Similarity Score for Phrase1 vs Phrase4 : {cosine_similarity(emb1,emb4)}")

Similarity Score for Phrase1 vs Phrase2 : 0.7828894853591919
Similarity Score for Phrase2 vs Phrase3 : 0.8803336024284363
Similarity Score for Phrase1 vs Phrase3 : 0.7153323888778687
Similarity Score for Phrase1 vs Phrase4 : 0.5174062252044678


In [10]:
import chromadb

# Creating persistent database in the current folder
client = chromadb.PersistentClient(path = "/content/chromadb")

collection = client.get_or_create_collection(
    name = "Basic_Questions",
    metadata = {"Description":"Practicing Basic Questions"}
)

print(f"Collection Created : {collection.name}")


Collection Created : Basic_Questions


In [11]:
print("Generating Embeddings for chunks and adding it to the collection")
chunk_embeddings = embedder.encode(chunks)

print(f"Generated {len(chunk_embeddings)} embeddings")
print("Adding chunks and embeddings to the collection")

collection.add(
    ids = [f"chunk_{i}" for i in range(len(chunks))],
    embeddings = chunk_embeddings.tolist(),
    documents = chunks,
    metadatas = [{f"chunk_index" : i} for i in range(len(chunks)) ]
    )
print(f"Added {collection.count()} chunks to database")

Generating Embeddings for chunks and adding it to the collection
Generated 62 embeddings
Adding chunks and embeddings to the collection
Added 62 chunks to database


In [12]:
def find_relevant_chunks(query, top_k = 5):
  query_embedding = embedder.encode(query)
  results = collection.query(
      query_embeddings = [query_embedding.tolist()],
      n_results = top_k,
      include = ["documents","metadatas"]

  )
  return results['documents'][0],results["metadatas"][0]

query = "Machine Learning"
chunks_found, metadata = find_relevant_chunks(query)

print(f"Query:{query}\n")
for i,(chunk,meta) in enumerate(zip(chunks_found , metadata)):
  print(f"---Result {i+1} (chunk {meta['chunk_index']}) ---")
  print(f"{chunk[:200]}...")
  print()

Query:Machine Learning

---Result 1 (chunk 40) ---
APIs and embed scores and recommendations without disrupting existing tools.  
 
• Maintaining Performance as Market Dynamics Changed:  Real estate markets shift fast due to regulations, seasonal 
tre...

---Result 2 (chunk 43) ---
stakeholders, data engineers, and analysts to define success metrics, review feature importance, and align the models 
with sales objectives. Regular sync -ups helped ensure that technical work stayed...

---Result 3 (chunk 2) ---
Texas, which strengthened my foundation in deep learning, MLOps, and scalable model 
deployment. 
 
Overall, I’m passionate about using AI to solve real -world business problems  — blending data 
engi...

---Result 4 (chunk 44) ---
--- Page 8 ---
 • Scikit-learn: I used Scikit -learn to develop baseline machine learning models, particularly for regression -based price 
prediction and lead scoring. Its intuitive APIs allowed for ...

---Result 5 (chunk 10) ---
transaction data withi

In [13]:
!pip install transformers accelerate bitsandbytes

In [15]:
from huggingface_hub import login
login(new_session=False)

In [16]:
from transformers import pipeline, BitsAndBytesConfig
import torch

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant = True
)

llm = pipeline(
    "text-generation",
    model = "meta-llama/Llama-3.1-8B-Instruct",
    model_kwargs = {'quantization_config':quantization_config},
    device_map = "auto"
)
print("Llama is loaded")

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Device set to use cuda:0


Llama is loaded


In [17]:
def answer_question(question,top_k=5):
  chunks_found,metadata = find_relevant_chunks(question,top_k = top_k)
  context = "\n\n".join(chunks_found)

  messages = [
      {
          "role":"system",
          "content":" You're a helpful assistant answering question about the give query. if context is has no relevant information, say so."

      },
      {
          "role":"user",
          "content":f"""Answer the following question based on the context provided : {context}
          Question :{question}
          Answer  based only on the context above"""
      }
  ]

  response = llm(messages,max_new_tokens=200, temperature=0.5,pad_token_id = llm.tokenizer.eos_token_id)
  answer = response[0]["generated_text"][-1]["content"]

  return {"answer": answer,"sources":metadata, "context_used": chunks_found}


In [18]:
result = answer_question("What is Machine Learning?")
print(result["answer"])

Based on the provided context, machine learning is a key concept that is mentioned multiple times. According to the text, machine learning involves developing models using Scikit-learn and XGBoost to solve real-world business problems, such as:

1. Price prediction: Developing supervised ML models to predict optimal property prices.
2. Lead scoring: Building ML models to score potential leads.
3. Fraud detection: Creating a real-time fraud detection engine.

The text highlights the importance of machine learning in driving measurable impact and improving decision-making. It also mentions the use of tools like Scikit-learn, XGBoost, and AWS EC2/S3 to build and deploy scalable ML solutions.

In this context, machine learning can be defined as the use of algorithms and statistical models to enable computers to learn from data, make predictions, and improve decision-making.


In [19]:
  !pip install streamlit

In [20]:
import streamlit as st
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import pipeline, BitsAndBytesConfig
import torch

st.set_page_config(
    page_title = "Custom ChatBot With RAG",
    layout="centered"

)

st.title("Custom Chatbot With RAG")
st.caption("ASK Questions")

@st.cache_resource # cache models so they load only once
def load_models():
  # Load embedder
  print("Loading embedding model....")
  embedder = SentenceTransformer('all-MiniLM-L6-v2')
  print("Embedding model is loaded")

  # Load chromadb client and collection
  client = chromadb.PersistentClient(path = "/content/chromadb")
  collection = client.get_or_create_collection(
      name = "Basic_Questions",
      metadata = {"Description":"Practicing Basic Questions"}
  )
  print(f"Collection Created : {collection.name}")

  # Load LLM
  quantization_config = BitsAndBytesConfig(
      load_in_4bit=True,
      bnb_4bit_quant_type="nf4",
      bnb_4bit_compute_dtype=torch.float16,
      bnb_4bit_use_double_quant = True
  )

  llm = pipeline(
      "text-generation",
      model = "meta-llama/Llama-3.1-8B-Instruct",
      model_kwargs = {'quantization_config':quantization_config},
      device_map = "auto"
  )
  print("Llama is loaded")
  return embedder, collection, llm


with st.spinner("Loading Model..."):
  embedder,collection,llm = load_models()

if "messages" not in st.session_state:
  st.session_state["messages"] = []

#Displaying Chat

for message in st.session_state.messages:
  with st.chat_message(message["role"]):
    st.markdown(message["content"])

# handling new input to chat
if prompt := st.chat_input("What is up?"):
  st.session_state.messages.append({"role":"user","content":prompt})
  with st.chat_message("user"):
    st.markdown(prompt)

  with st.spinner("Generating Response..."):
    result = answer_question(prompt)
    answer = result["answer"]
    sources = result["sources"]
    st.markdown(answer) # Corrected indentation
  st.session_state.messages.append({"role":"assistant","content":result["answer"]})



with st.sidebar:
  st.title("RAG Settings")
  st.markdown("**Retrieval Top-K**")

  if st.button("Reset Chat"):
    st.session_state["messages"] = []
    st.rerun()

2026-01-10 03:14:18.945 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-10 03:14:18.946 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-10 03:14:19.186 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-01-10 03:14:19.187 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-10 03:14:19.189 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-10 03:14:19.190 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-10 03:14:19.192 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

Loading embedding model....


2026-01-10 03:14:19.701 Thread 'Thread-5': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-10 03:14:19.703 Thread 'Thread-5': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-10 03:14:19.704 Thread 'Thread-5': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-10 03:14:19.706 Thread 'Thread-6': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-10 03:14:19.707 Thread 'Thread-6': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-10 03:14:19.709 Thread 'Thread-6': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Embedding model is loaded
Collection Created : Basic_Questions


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
2026-01-10 03:16:07.678 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-10 03:16:07.679 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-10 03:16:07.680 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-10 03:16:07.722 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-10 03:16:07.723 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-10 03:16:07.725 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-10 03:16:07.727 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-10 03:16:07.728 Session state does not function when running a script without `stream

Llama is loaded
